In [2]:
import sklearn
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report , accuracy_score
import numpy as np

df = pd.read_csv('insurance_premium_dataset.csv')

df.sample(5)

df_feat = df.copy()

df['occupation'].unique()

# Convert smoker Yes/No to True/False
df_feat["smoker"] = (
    df_feat["smoker"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": True,
        "no": False
    })
)

#print(df_feat["lifestyle_risk"].value_counts())
print(df_feat["smoker"].value_counts())

print(df_feat["smoker"].unique())

#Features 1: BMI
df_feat['bmi'] = (
    df_feat["weight"]/
    ((df_feat["height"]/100)** 2)
)

#Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "Young"
    elif age < 45:
        return "Adult"
    elif age < 60:
        return "middle_aged"
    else:
        return "Senior"

df_feat["age_group"] = df_feat["age"].apply(age_group)

# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

df_feat["lifestyle_risk"] = df_feat.apply(
    lifestyle_risk,
    axis=1
)

print(df_feat["bmi"].describe())

tier_1_cities = ["Mumbai", "Delhi", "Banglore", "Chennai", "Kolkata", "Hyperabad", "Pune"]
tier_2_cities = [
    "jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore", 
    "Bhopal", "Nagpur", "Vadodra", "Surat", "Jodhpur", "Raipur", "Amritsar", "Varanasi","Agra",
    "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvantpuram", "Ludhiana", "Nashik", "Allahabad", 
    "Udaipur", "Aurangaba", "Hubli", "Belgaum","Salwm", "Vijaywada", "Triuchirappali", "Bhavanagr", "gwalior", 
    "Dhanbad", "Bareily", "Aligarh", "Gaya", "Kozhikoda", "warangal", "Kolhapur", "Bilahpur", "Jalandhar", "Noida",
    "Guntur", "Asansol", "Silliguri"
    ]

# Feature 4: City tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

df_feat["city_tier"] = df_feat["city"].apply(city_tier)

df_feat.drop(columns=['age','weight','height','smoker','city'])[['income_lpa','occupation','bmi','age_group','lifestyle_risk','city_tier']]

# Select features and target
x = df_feat[['bmi','age_group','lifestyle_risk','city_tier','income_lpa','occupation']]
y = df_feat['insurance_premium_category']

# Define categorical and numeric features
categorical_features = ['age_group','lifestyle_risk','occupation','city_tier']
numeric_features = ['bmi','income_lpa']

# Create column transformer for OHE
preprocessor = ColumnTransformer(
   transformers=[
       ("cat",OneHotEncoder(handle_unknown='ignore'),categorical_features),
       ('num','passthrough',numeric_features)
   ]
)

# Create a piprline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(x, y,test_size=0.2,random_state = 1)
pipeline.fit(X_train,y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_pred))

print(
    classification_report(
        y_test,
        y_pred
    )
)

X_test.sample(5)

import pickle



encoder = (
    pipeline
    .named_steps["preprocessor"]
    .named_transformers_["cat"]
)

for feature, categories in zip(
    categorical_features,
    encoder.categories_
):
    print(feature, ":", categories)

# save the trained pipeline using pickle
pickle_model_path =  "Model.pkl"
with open(pickle_model_path, "wb") as f:
   pickle.dump(pipeline,f)






smoker
False    784
True     216
Name: count, dtype: int64
[False  True]
count    1000.000000
mean       18.377002
std         2.699744
min        11.879218
25%        16.358531
50%        18.179081
75%        20.148624
max        28.512192
Name: bmi, dtype: float64
Accuracy: 0.94
              precision    recall  f1-score   support

         Low       0.96      0.97      0.97       173
      Medium       0.80      0.74      0.77        27

    accuracy                           0.94       200
   macro avg       0.88      0.86      0.87       200
weighted avg       0.94      0.94      0.94       200

age_group : ['Adult' 'Senior' 'Young' 'middle_aged']
lifestyle_risk : ['low' 'medium']
occupation : ['Accountant' 'Business' 'Doctor' 'Engineer' 'Government Employee'
 'Manager' 'Other' 'Software Developer' 'Student' 'Teacher']
city_tier : [1 2 3]
